<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_6/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_1_%D0%9E%D1%81%D0%BD%D0%BE%D0%B2%D1%8B_%D1%81%D0%BE%D0%B7%D0%B4%D0%B0%D0%BD%D0%B8%D1%8F_RAG_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D0%B0_%D0%BD%D0%B0_%D0%BB%D0%BE%D0%BA%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D0%B9_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.1. Основы создания RAG-агента на локальной LLM  


## Введение: почему мы строим именно так?

Представьте, что вы даёте задание умному помощнику: *«Найди в моих документах ответ на вопрос и объясни его»*. Обычный чат‑бот, даже самый продвинутый, не сможет этого сделать – он знает только то, что было в его обучающих данных. Наш же агент будет **сам искать нужную информацию в вашем личном хранилище**, а затем формулировать ответ, опираясь на неё. Это и есть **RAG (Retrieval-Augmented Generation)** – технология, которая сочетает поиск и генерацию.

В этой лекции мы пройдём **восемь шагов** – от установки необходимого ПО до создания агента, который умеет **решать, когда искать**, а когда отвечать из собственных знаний. Все шаги мы будем выполнять **локально**, без облачных API, используя **Ollama** и модель **Qwen 2.5 3B**. Это даёт нам полный контроль и нулевые финансовые затраты.

> **Цель лекции** – не просто дать готовый код, а объяснить **каждый этап**: зачем мы это делаем, как это работает внутри и какие подводные камни нас ждут.

---

## Шаг 0. Что такое ИИ-агент и RAG?

Прежде чем писать код, давайте договоримся о терминах.

- **ИИ-агент** – это программа, которая **в цикле** принимает решения, используя большую языковую модель (LLM) как «мозг». Агент может вызывать внешние инструменты (поиск, калькулятор, API), анализировать результаты и повторять действия до достижения цели.
- **RAG** – это частный случай агентского поведения, когда инструментом является **поиск по документам**. Модель получает запрос, находит релевантные фрагменты из базы знаний и генерирует ответ на их основе.

Почему RAG так важен? Потому что LLM часто «галлюцинирует» – придумывает факты. Если мы **передаём ей точные выдержки из документов**, мы сильно снижаем риск ошибок и делаем ответы проверяемыми.

В нашем проекте мы построим агента, который:
1. Принимает вопрос.
2. Решает, нужен ли поиск (или ответить сразу).
3. Если нужен – ищет в документах семантически близкие фрагменты.
4. Передаёт их модели вместе с вопросом.
5. Возвращает пользователю итоговый ответ.

---

## Шаг 1. Установка Ollama и загрузка модели

**Ollama** – это программа, которая позволяет запускать большие языковые модели на вашем компьютере. Она работает как локальный сервер, и мы будем обращаться к нему через HTTP‑запросы.

### 1.1. Скачиваем и устанавливаем
Перейдите на официальный сайт [ollama.com](https://ollama.com) и скачайте установщик для вашей операционной системы (Windows, macOS, Linux). Установка стандартная – просто следуйте инструкциям.

После установки откройте терминал (командную строку) и проверьте, что Ollama работает:
```bash
ollama --version
```
Если вы видите номер версии – всё хорошо.

### 1.2. Загружаем модель Qwen 2.5 3B
Модель **Qwen 2.5 3B** – это легковесная, но достаточно умная языковая модель от Alibaba. Она весит около 2 ГБ и отлично подходит для первых экспериментов. Загружаем её командой:
```bash
ollama pull qwen2.5:3b
```
Процесс займёт несколько минут в зависимости от скорости интернета.




```
PS D:\Science\AI_Agent_Demo> ollama pull qwen2.5:3b                                                                                        
pulling manifest
pulling 5ee4f07cdb9b: 100% ▕█████████████████████████████████████████████████████████████████████████████▏ 1.9 GB                         
pulling 66b9ea09bd5b: 100% ▕█████████████████████████████████████████████████████████████████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕█████████████████████████████████████████████████████████████████████████████▏ 1.5 KB                         
pulling b5c0e5cf74cf: 100% ▕█████████████████████████████████████████████████████████████████████████████▏ 7.4 KB                         
pulling 161ddde4c9cd: 100% ▕█████████████████████████████████████████████████████████████████████████████▏  487 B                         
verifying sha256 digest
writing manifest
success
PS D:\Science\AI_Agent_Demo>
```





### 1.3. Проверяем работу модели
Выполните команду:
```bash
ollama run qwen2.5:3b
```
После этого вы попадёте в интерактивный режим, где можете написать `Привет` и получить ответ. Если ответ появился – модель работает.

> **Важно:** Ollama должен быть запущен всё время, пока мы разрабатываем. Обычно он автоматически стартует как служба, но если что – просто держите терминал с `ollama serve` открытым.





```
PS D:\Science\AI_Agent_Demo> ollama run qwen2.5:3b
>>> 2+2=?
2 + 2 equals 4.

>>> Send a message (/? for help)
```





## Шаг 2. Установка необходимых библиотек

Для нашего проекта нам понадобится Python (версия 3.8+) и всего одна библиотека на первом этапе – `requests`, чтобы отправлять HTTP‑запросы к Ollama.

Создайте папку для проекта, например `my_first_rag`, и внутри неё установите:
```bash
pip install requests
```
Позже мы добавим `sentence-transformers` и `scikit-learn` для эмбеддингов, но пока остановимся на этом.

---

## Шаг 3. Первый запрос к модели – проверка связи

Прежде чем строить сложные конструкции, убедимся, что мы умеем общаться с моделью из Python. Создайте файл `test_llm.py` со следующим содержимым:

```python
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": "Что такое Python?",
        "stream": False
    }
)

print(response.json()["response"])
```

**Разбор кода:**
- Мы отправляем POST‑запрос на эндпоинт `/api/generate` локального сервера Ollama.
- В JSON указываем модель, промпт и отключаем потоковую передачу (`stream: False`), чтобы получить ответ целиком.
- Из ответа извлекаем поле `response` и печатаем его.

Запустите:
```bash
python test_llm.py
```
Если вы видите осмысленный ответ о Python – значит, связь установлена, и мы готовы двигаться дальше.

**Что мы сейчас сделали?** Мы научились программно вызывать LLM. Это базовый кирпичик, на котором будет построен весь остальной функционал.





```
PS D:\Science\AI_Agent_Demo> python test_llm.py
Python - это высокоуровневый язык программирования с простым и гибким синтаксисом. Он был разработан Брэнданом Леви в середине 1980-х годов и открыт для публичного использования в 1991 году. Python является интерпретируемым языком, что позволяет быстро создавать программное обеспечение без необходимости компилирования кода.

Ключевые особенности Python:

1. Объектно-ориентированность: Поддерживает принципы объектного программирования.
2. Простота и простота: Легко изучается, благодаря своему понятному синтаксису.
3. Гибкость: Может использоваться для создания различных типов программных продуктов, от скриптов до полноценных приложений или систем.
4. Мультипарадигмальный: Поддерживает процедурное программирование, объектно-ориентированное и функциональное программирование.
5. Интерпретируемый язык: Программы могут быть написаны в режиме реального времени, что увеличивает скорость разработки.

Python используется для разработки различных приложений, таких как:

- Web-разработка
- Мобильная разработка (например, Android)
- Дизайн библиотек и инструментов
- Графический интерфейс (GUI) приложений
- Автоматизация задач в операционных системах
- Анализ данных

Python также активно используется в образовательной среде для преподавания программированию из-за своего простого синтаксиса и легкости использования.
PS D:\Science\AI_Agent_Demo>
```





## Шаг 4. Первый RAG без поиска (просто подстановка документа)

Теперь представим, что у нас есть один большой документ, и мы хотим, чтобы модель отвечала на вопросы **только на основе этого документа**. Это самый простой способ внедрить знания – просто вставить текст документа в промпт.

Создайте файл `simple_rag.py`:

```python
import requests

document = """
Python является языком программирования.

RAG (Retrieval Augmented Generation) позволяет находить информацию
в документах и передавать найденный текст модели.

Ollama позволяет запускать локальные языковые модели.
"""

question = input("Введите вопрос: ")

prompt = f"""
Используй только информацию из документа.

Документ:

{document}

Вопрос:
{question}
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False
    }
)

print("\nОтвет:")
print(response.json()["response"])
```

**Что здесь происходит?**
- Мы жёстко задаём документ (трёхстрочный текст).
- Формируем промпт, который инструктирует модель **строго** использовать только этот документ.
- Отправляем запрос и печатаем ответ.

**Проверка:** запустите и введите вопрос, например `Что такое RAG?`. Модель должна ответить, используя информацию из документа. Если вопрос не касается документа (например, `Как погода?`), модель всё равно попытается ответить, но она ограничена документом – это хорошо, но негибко.

**Почему это ещё не RAG?** Потому что мы не ищем релевантную часть документа – мы отдаём весь документ целиком. Если документ станет большим (сотни страниц), модель не сможет обработать такой длинный контекст. Поэтому нам нужен **поиск**.




```
PS D:\Science\AI_Agent_Demo> python .\simple_rag.py
Введите вопрос: Что такое RAG?

Ответ:
РAG (Retrieval-Augmented Generation) - это технология, которая позволяет системе находить информацию в ранее загруженных документах и затем использовать эту найденную информацию для генерации нового контента. Такой подход позволяет улучшить точность и полноту ответов, основываясь на уже существующей информации.
PS D:\Science\AI_Agent_Demo> python .\simple_rag.py
Введите вопрос: Как погода?

Ответ:
Я не могу найти информацию о погоде в данном документе. Этот документ подразумевает контекст по поводу языковых моделей Python и RAG (Retrieval Augmented Generation), а также о возможности запуска локальных языковых моделей с помощью Ollama, но не содержит информации об текущей погоде.
PS D:\Science\AI_Agent_Demo>
```





## Шаг 5. Разбиваем документ на чанки (куски)

Чтобы эффективно искать, нужно разбить большой текст на небольшие фрагменты – **чанки**. В идеале каждый чанк должен быть смысловым блоком (абзацем или несколькими предложениями). Пока мы просто создадим список строк.

Создайте `chunk_rag.py`:

```python
import requests

chunks = [
    "Python является языком программирования.",
    "RAG позволяет искать информацию в документах.",
    "Ollama запускает локальные модели.",
    "LangChain помогает создавать агентов."
]

question = input("Вопрос: ")

context = "\n".join(chunks)   # склеиваем все чанки

prompt = f"""
Контекст:

{context}

Ответь только используя контекст.

Вопрос:
{question}
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False
    }
)

print(response.json()["response"])
```

Теперь у нас есть отдельные куски, но мы по‑прежнему отправляем **все** чанки модели – мы не выбираем релевантные. Это просто переходная форма. Однако заметьте: если мы будем добавлять чанки, контекст будет расти, и модель может «забыть» начало. Поэтому нужно переходить к **выборочному поиску**.





```
PS D:\Science\AI_Agent_Demo> python chunk_rag.py   
Вопрос: Как погода?
Я не могу ответить на вопрос "Как погода?" в рамках предоставленного контекста, так как информация о погоде не связана ни с Python, ни с RAG, ни с Ollama, ни с LangChain. Эти термины относятся к разным областям: Python - это языковой среда для программирования; RAG и LangChain вовлекаются в области искусственного интеллекта и машинного обучения, а Ollama имеет отношение к запуску локальных моделей. Погода, как правило, связана с наукою или через специализированные сервисы по прогнозированию погоды.
PS D:\Science\AI_Agent_Demo> python chunk_rag.py
Вопрос: Что такое RAG?
RAG (Relevant AI Retrieval-Augmented Generation) - это метод или подход, который позволяет искать и использовать информацию из документов при генерации текста. Это часть технологий машинного обучения, где модели могут получать дополнительное обогащение данных из внешних источников для улучшения своей работы.
PS D:\Science\AI_Agent_Demo>

```




## Шаг 6. Настоящий семантический поиск – эмбеддинги и косинусная близость

Теперь мы подходим к сердцу RAG – **семантическому поиску**. Идея проста:
1. Каждый чанк превращаем в вектор (эмбеддинг) с помощью специальной модели, которая кодирует смысл текста.
2. Вопрос пользователя тоже превращаем в вектор.
3. Вычисляем косинусное расстояние между вектором вопроса и векторами всех чанков.
4. Выбираем чанк с наибольшей похожестью – он и будет релевантным.

Для этого нам понадобятся библиотеки:
```bash
pip install sentence-transformers scikit-learn
```

`SentenceTransformers` даёт готовые модели для получения эмбеддингов предложений. Мы будем использовать лёгкую модель `all-MiniLM-L6-v2`.

Создайте `semantic_rag.py`:

```python
import requests
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Наши чанки (можно расширить)
chunks = [
    "Python является языком программирования.",
    "RAG позволяет искать информацию в документах.",
    "Ollama запускает локальные модели.",
    "LangChain помогает создавать агентов."
]

# Загружаем модель для эмбеддингов (скачается автоматически при первом запуске)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Заранее вычисляем эмбеддинги для всех чанков
chunk_embeddings = embedder.encode(chunks)

question = input("Вопрос: ")

# Эмбеддинг вопроса
question_embedding = embedder.encode([question])

# Считаем косинусную близость между вопросом и каждым чанком
scores = cosine_similarity(question_embedding, chunk_embeddings)[0]
# scores – массив чисел от -1 до 1, чем ближе к 1, тем похожее

# Находим индекс чанка с максимальной близостью
best_index = scores.argmax()

# Извлекаем лучший чанк
best_chunk = chunks[best_index]

print("\nНайденный текст:")
print(best_chunk)

# Формируем промпт с этим чанком
prompt = f"""
Контекст:

{best_chunk}

Ответь на вопрос, используя только этот контекст.

Вопрос:
{question}
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False
    }
)

print("\nОтвет:")
print(response.json()["response"])
```

**Анатомия шага и важное ограничение:**
- Модель эмбеддингов превращает текст в вектор размерности 384 (для `all-MiniLM-L6-v2`).
- Мы заранее вычисляем и сохраняем эмбеддинги чанков — в реальных проектах для этого используют векторные базы данных.
- Косинусная близость измеряет смысловое сходство: чем ближе к 1, тем релевантнее чанк.
- **Мы берём только один лучший чанк** — это наглядно, но ненадёжно. Если ответ требует данных из нескольких фрагментов, модель его не получит.

> **Почему одного чанка недостаточно.** Допустим, один чанк гласит: «Python создал Гвидо ван Россум», а другой — «Первый релиз Python вышел в 1991 году». Вопрос «Когда и кем был создан Python?» требует обоих. Поэтому в реальных RAG-системах используют **top‑K**: отбирают несколько самых похожих чанков и объединяют в общий контекст.

Мы применим это улучшение в следующем шаге — функция `search_docs` будет возвращать уже `top_k=3` чанка. А пока можете убедиться в полезности нескольких чанков: добавьте после строки `scores = ...` такой код и посмотрите на оценки трёх лучших фрагментов:

```python
# Выведем топ-3 чанка для наглядности
top_indices = scores.argsort()[-3:][::-1]
print("\nТоп-3 чанка по релевантности:")
for i in top_indices:
    print(f"[{scores[i]:.2f}] {chunks[i]}")
```

Теперь у нас есть **настоящий семантический поиск**, и мы понимаем его текущее ограничение. Проверьте: задайте вопрос `Что такое RAG?` — должен победить чанк про RAG. Помните: одного чанка достаточно для демонстрации, но для надёжного агента мы перейдём на несколько.
```



```
PS D:\Science\AI_Agent_Demo> python semantic_rag.py
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13570.39it/s]
Вопрос: Что такое RAG?

Топ-3 чанка по релевантности:
[0.69] RAG позволяет искать информацию в документах.
[0.55] LangChain помогает создавать агентов.
[0.47] Ollama запускает локальные модели.

Найденный текст:
RAG позволяет искать информацию в документах.

Ответ:
РAG (Retrieval-Augmented Generation) - это подход, который позволяет использовать результаты поиска информации из документов для дополнения и улучшения генерируемого текста.
PS D:\Science\AI_Agent_Demo>

```



Да, давайте полностью переработаем Шаг 7, чтобы он честно показывал настоящую, пусть и простую, маршрутизацию. Вот исправленный вариант – и код, и описание.

---

## Шаг 7. Превращаем RAG в агента – добавляем маршрутизацию

Агент должен **сам решать**, когда использовать поиск по документам, а когда ответить, опираясь только на свои общие знания. Самый простой и наглядный способ – проверить, есть ли в вопросе ключевые слова, связанные с темами наших документов. Если есть – ищем в чанках, если нет – отвечаем без них.

> В реальных агентах решение обычно принимает сама языковая модель, анализируя запрос, но для первого знакомства нам достаточно простого условия. Такой подход сразу демонстрирует идею **маршрутизации** – выбора одного из нескольких путей обработки.

Создайте файл `agent.py` со следующим содержимым:

```python
import requests
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. База знаний в виде чанков
chunks = [
    "Python является языком программирования высокого уровня.",
    "RAG (Retrieval-Augmented Generation) — это метод, который позволяет находить информацию в документах и использовать её для генерации ответов с помощью языковых моделей.",
    "Ollama — это инструмент для запуска локальных языковых моделей, таких как Qwen, Llama и другие.",
    "LangChain — это фреймворк для создания приложений на основе LLM, включая агентов и цепочки.",
]

# 2. Загружаем модель эмбеддингов
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks)

def search_docs(query, top_k=3):
    """Возвращает top_k наиболее релевантных чанков по запросу"""
    query_emb = embedder.encode([query])
    scores = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_indices = scores.argsort()[-top_k:][::-1]
    return "\n".join([chunks[i] for i in top_indices]), top_indices

# 3. Основной цикл агента
question = input("Вопрос: ")

# 4. МАРШРУТИЗАЦИЯ: решаем, нужен ли поиск
#    Если вопрос касается тем из нашей базы – ищем, иначе – нет
topics = ["python", "rag", "ollama", "langchain", "язык программирования",
          "поиск", "документ", "локальная модель", "агент"]
need_search = any(topic in question.lower() for topic in topics)

if need_search:
    # Ищем релевантные чанки и формируем контекстный промпт
    context, indices = search_docs(question)
    prompt = f"""
Ты — строгий помощник. Отвечай ТОЛЬКО на основе приведённого КОНТЕКСТА.
НЕ используй свои знания.
Если в контексте нет прямого ответа на вопрос, скажи: "В документах нет информации по этому вопросу."

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ (только из контекста, кратко и по существу):
"""
else:
    # Контекст не требуется, модель отвечает из общих знаний
    prompt = f"""
Ответь на вопрос, используя свои общие знания. Будь краток.

ВОПРОС:
{question}

ОТВЕТ:
"""

# 5. Вызов LLM
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False
    }
)

print("\nОтвет:")
print(response.json()["response"])
```

**Что изменилось и почему это важно:**

- **Добавлено настоящее ветвление.** Агент теперь не ищет чанки на каждый вопрос. Если спросить, например, *«Какая столица Франции?»*, поиск не выполнится, модель ответит из своих знаний.
- **Список ключевых тем** `topics` связывает вопросы с нашей базой. Вы можете легко его расширять, добавляя новые слова, когда пополняете документы.
- **Два разных промпта** подчёркивают разницу в поведении: в одном случае модель строго ограничена контекстом, в другом — свободна.
- Теперь утверждение *«агент анализирует вопрос и выбирает действие»* полностью соответствует коду.

**Проверьте работу:**

- Вопрос: *«Что такое RAG?»* – агент найдёт чанк про RAG и ответит по нему.
- Вопрос: *«Сколько будет 2+2?»* – агент не найдёт ключевых слов, пропустит поиск и ответит, используя свои знания.
- Вопрос: *«Расскажи про Ollama»* – поиск сработает, и ответ будет основан на чанке.

**Что мы получили?**  
Мы построили элементарного агента с **маршрутизацией на основе ключевых слов**. Это уже не «всегда искать», а осознанный выбор инструмента. В следующей лекции мы научим модель саму принимать такое решение, анализируя запрос без заранее заданного списка.




```
PS D:\Science\AI_Agent_Demo> python agent.py   
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 14188.56it/s]
Вопрос: Что такое RAG?

Ответ:
RAG — это метод, который использует документы для получения информации для генерации ответов с помощью языковых моделей.
PS D:\Science\AI_Agent_Demo> python agent.py
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 12943.06it/s]
Вопрос: Сколько будет 2+2?

Ответ:
2+2=4
PS D:\Science\AI_Agent_Demo> python agent.py   
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13529.17it/s]
Вопрос: Расскажи про Ollama

Ответ:
Ollama — это инструмент для запуска локальных языковых моделей, такие как Qwen, Llama и другие.
PS D:\Science\AI_Agent_Demo>
```



## Шаг 8. Куда двигаться дальше? Следующая ступень

Мы построили работающего агента с RAG. Но он пока игрушечный. Для реальных задач нужно:

1. **Заменить `knowledge` на реальные документы** – PDF, Excel, базы данных, веб‑страницы.
2. **Использовать векторную базу данных** (Chroma, FAISS) для хранения миллионов чанков и быстрого поиска.
3. **Добавить больше инструментов** – например, `calculator`, `web_search`, `send_email`.
4. **Перейти на фреймворки** – **LangChain** упростит интеграцию, а **LangGraph** даст строгий контроль над циклом и поддержку человеческого вмешательства.
5. **Улучшить память** – хранить историю диалога, чтобы агент помнил предыдущие вопросы.
6. **Обеспечить наблюдаемость** – логировать каждый шаг, чтобы отлаживать и улучшать систему.

Установите необходимые пакеты:
```bash
pip install langchain langgraph chromadb
```
и начните заменять простую строку на загрузку документов из папки.

---

## Заключение первой лекции

Мы прошли путь от установки Ollama до создания агента, который умеет искать информацию в документах и отвечать на вопросы. Каждый шаг был осмысленным и постепенным. Теперь вы понимаете:

- Как работает локальная LLM и как с ней общаться.
- Что такое эмбеддинги и семантический поиск.
- Как построить простейший RAG и превратить его в агента с маршрутизацией.

Этот фундамент позволит вам двигаться к более сложным системам, не теряя понимания происходящего «под капотом». В следующей лекции мы займёмся масштабированием: подключим векторную базу, реализуем многокусочный контекст и добавим память. А пока – экспериментируйте с кодом, меняйте чанки, пробуйте разные вопросы и наблюдайте, как агент принимает решения.





## Домашнее задание

### Обязательная часть (для всех)

1. **Топ‑3 чанка в контексте.**  
   Модифицируйте `semantic_rag.py` так, чтобы модель получала не один лучший чанк, а три самых релевантных, объединённых в общий контекст (через `"\n"`).  
   **Что сравнить:**  
   - Задайте **один сложный вопрос**, требующий информации из разных частей документа (например: «Что такое RAG и как он связан с Ollama?»).  
   - Сравните ответы с вариантом из лекции (один чанк) — в каком случае ответ полнее и точнее?  
   - Запишите оба ответа в отчёт и сделайте вывод.

2. **Реальные документы.**  
   - Создайте текстовый файл (`.txt`) с **5–10 абзацами** на любую тему (например, статья из Википедии, выдержка из книги, описание технологии).  
   - Напишите скрипт, который загружает файл, разбивает его на чанки по двойным переносам строк (`\n\n`) и подставляет полученный список в `chunks`.  
   - Убедитесь, что вы удалили пустые строки: `[chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]`.  
   - Задайте **3 вопроса** по вашему документу и проверьте, находит ли агент правильные фрагменты.  
   - Если ответ неудовлетворительный — попробуйте другой способ разбиения (например, по точкам с ограничением длины).



### Дополнительная часть (эксперименты — выполните минимум 3 из 5)

3. **Другая модель эмбеддингов.**  
   Замените `all-MiniLM-L6-v2` на любую из [sentence-transformers](https://sbert.net/docs/pretrained_models.html):  
   - `paraphrase-multilingual-MiniLM-L12-v2` — лёгкая мультиязычная;  
   - `intfloat/multilingual-e5-small` или `intfloat/multilingual-e5-large` — современная, но `e5-large` большая (~1.3 ГБ);  
   - `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` — качественная, но тяжёлая.  
   **Что сравнить:**  
   - Выведите для одного запроса **оценки релевантности всех чанков** (массив `scores`).  
   - Сравните, изменился ли порядок чанков. Какая модель лучше улавливает смысл на русском языке?  
   - Засеките время загрузки модели и инференса (через `time.time()`).

4. **Другая метрика сходства.**  
   Вместо косинусной близости используйте:  
   - Евклидово расстояние (`from sklearn.metrics.pairwise import euclidean_distances`). **Важно:** чем **меньше** расстояние, тем ближе векторы — не забудьте инвертировать или сортировать по возрастанию.  
   - Скалярное произведение — для его использования векторы нужно **нормализовать** (разделить на длину), иначе длинные векторы будут давать большие значения.  
   **Что сравнить:**  
   - Меняется ли порядок чанков при замене метрики?  
   - Влияет ли это на итоговый ответ модели? (задайте вопрос, где порядок важен).

5. **Разные LLM (основной эксперимент).**  
   Через `ollama pull` скачайте несколько моделей разного размера и архитектуры:  
   ```bash
   ollama pull deepseek-r1:1.5b
   ollama pull deepseek-r1:7b
   ollama pull qwen2.5:7b
   ollama pull llama3.1:8b
   ollama pull phi3:mini
   ollama pull mistral:7b   # опционально
   ```  
   > **Обратите внимание:** модели 7B–8B требуют 6–8 ГБ ОЗУ/VRAM. Если ваш компьютер ограничен, используйте версии 1.5B–3B или выполняйте эксперименты по одной модели за раз.

   Для каждой модели:  
   - Замените `"qwen2.5:3b"` в `agent.py` на новое имя.  
   - Задайте **один и тот же набор из 3 вопросов** (1 по вашим документам, 1 общий, 1 сложный).  
   - Засеките время ответа (вставьте `start_time = time.time()` до запроса и `print(time.time() - start_time)` после).  
   - Оцените:  
     - **Точность:** соответствует ли ответ контексту?  
     - **Полноту:** не упускает ли модель важные детали?  
     - **Строгость следования инструкции:** начинает ли модель «галлюцинировать» или добавлять свои знания?  
     - **Скорость:** сколько секунд требуется на ответ.  
   > **Важное замечание про DeepSeek-R1:** эта модель «рассуждающая». Она может выдавать длинный внутренний монолог (`<think>...</think>`) перед ответом. Чтобы получить только финальный ответ, добавьте в промпт инструкцию: `"Не показывай свои рассуждения, дай только краткий ответ."` Или используйте парсинг, обрезая текст после `</think>`.

6. **Маршрутизация через LLM.**  
   В `agent.py` замените проверку ключевых слов на интеллектуальное решение:  
   - Отправьте вопрос модели с просьбой вернуть JSON `{"use_search": true/false}`.  
   - Парсите ответ и используйте его для ветвления (используйте `json.loads()`).  
   - **Совет:** чтобы избежать сбоев при парсинге, оберните `json.loads()` в `try/except` и предусмотрите поведение по умолчанию (например, всегда выполнять поиск), если JSON не распарсился.  
   **Что сравнить:**  
   - Насколько гибче стал агент? Приведите примеры вопросов, где модель сама понимает, нужен ли поиск, даже если тема не упомянута в списке `topics`.  
   - Возникают ли ошибки при парсинге JSON? Как их можно обработать?



### Формат сдачи
- Загрузите код каждого выполненного пункта в отдельную папку (или используйте Git).  
- Приложите **отчёт** (`.md` или `.pdf`) с выводами по каждому сравнению: что ожидали, что получили, какие модели/методы сработали лучше и почему.

### Критерии оценки
- **Обязательная часть (max 5 баллов):**  
  - Корректная реализация top‑3 (1 балл) + сравнение с одним чанком (1 балл).  
  - Работа с реальным файлом (2 балла) + анализ качества (1 балл).  
- **Дополнительная часть (каждый выполненный пункт — до 2 баллов, максимум +10 баллов):**  
  - 1 балл за реализацию, 1 балл за осмысленные выводы.  
- **Максимум за ДЗ: 15 баллов.**

